# YOLOv8s Object Detection Training
Training on dataset_subsample_top10 dataset (Pascal VOC to YOLO conversion)

In [ ]:
# Install required packages
%pip install ultralytics opencv-python matplotlib

In [ ]:
import xml.etree.ElementTree as ET
import shutil
from pathlib import Path
import yaml
import random
import torch
import os

## Configuration

In [ ]:
# Check CUDA availability
CUDA_AVAILABLE = torch.cuda.is_available()
print(f'CUDA available: {CUDA_AVAILABLE}')
if CUDA_AVAILABLE:
    print(f'CUDA device count: {torch.cuda.device_count()}')
    print(f'CUDA device name: {torch.cuda.get_device_name(0)}')

# Set training device
TRAINING_DEVICE = 0 if CUDA_AVAILABLE else 'cpu'
print(f'Training device: {TRAINING_DEVICE}')

# Original dataset path (Pascal VOC format)
ORIGINAL_DATASET_PATH = Path('data/dataset_subsample_top10')

# Converted dataset path (YOLO format) - will be created
YOLO_DATASET_PATH = Path('data/dataset_subsample_top10_yolo')

# Model configuration
MODEL_NAME = 'yolov8s.pt'
EPOCHS = 10
BATCH_SIZE = 8
IMGSZ = 640

# Output directory for training results
OUTPUT_DIR = Path('runs/detect')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Random seed for reproducibility
SEED = 42
random.seed(SEED)

## Dataset Format Detection

In [ ]:
print(f'Original dataset path: {ORIGINAL_DATASET_PATH}')
print(f'Dataset exists: {ORIGINAL_DATASET_PATH.exists()}')

if ORIGINAL_DATASET_PATH.exists():
    annotations_dir = ORIGINAL_DATASET_PATH / 'annotations'
    images_dir = ORIGINAL_DATASET_PATH / 'images'
    
    print(f'Annotations directory exists: {annotations_dir.exists()}')
    print(f'Images directory exists: {images_dir.exists()}')
    
    if annotations_dir.exists():
        num_annotations = len(list(annotations_dir.glob('*.xml')))
        print(f'Number of annotation files (Pascal VOC): {num_annotations}')
    
    if images_dir.exists():
        num_images = len(list(images_dir.glob('*.jpg'))) + len(list(images_dir.glob('*.png')))
        print(f'Number of images: {num_images}')

print(f'YOLO dataset path: {YOLO_DATASET_PATH}')
print(f'YOLO dataset exists: {YOLO_DATASET_PATH.exists()}')

## Convert Pascal VOC to YOLO Format

In [ ]:
def voc_to_yolo(bbox, img_width, img_height):
    xmin, ymin, xmax, ymax = bbox
    width = xmax - xmin
    height = ymax - ymin
    x_center = xmin + width / 2
    y_center = ymin + height / 2
    x_center_norm = x_center / img_width
    y_center_norm = y_center / img_height
    width_norm = width / img_width
    height_norm = height / img_height
    return x_center_norm, y_center_norm, width_norm, height_norm

In [ ]:
def convert_voc_to_yolo(original_dataset_path, yolo_dataset_path, test_split=0.2, val_split=0.1):
    annotations_dir = original_dataset_path / 'annotations'
    images_dir = original_dataset_path / 'images'
    
    (yolo_dataset_path / 'images' / 'train').mkdir(parents=True, exist_ok=True)
    (yolo_dataset_path / 'images' / 'val').mkdir(parents=True, exist_ok=True)
    (yolo_dataset_path / 'images' / 'test').mkdir(parents=True, exist_ok=True)
    (yolo_dataset_path / 'labels' / 'train').mkdir(parents=True, exist_ok=True)
    (yolo_dataset_path / 'labels' / 'val').mkdir(parents=True, exist_ok=True)
    (yolo_dataset_path / 'labels' / 'test').mkdir(parents=True, exist_ok=True)
    
    annotation_files = list(annotations_dir.glob('*.xml'))
    print(f'Found {len(annotation_files)} annotation files')
    
    all_classes = set()
    for ann_file in annotation_files:
        tree = ET.parse(ann_file)
        root = tree.getroot()
        for obj in root.findall('object'):
            class_name = obj.find('name').text
            all_classes.add(class_name)
    
    sorted_classes = sorted(all_classes)
    class_to_idx = {cls_name: idx for idx, cls_name in enumerate(sorted_classes)}
    idx_to_class = sorted_classes
    
    print(f'Found {len(sorted_classes)} unique classes')
    print(f'Sample classes: {sorted_classes[:5]}...')
    
    random.shuffle(annotation_files)
    
    num_files = len(annotation_files)
    num_test = int(num_files * test_split)
    num_val = int(num_files * val_split)
    num_train = num_files - num_test - num_val
    
    print(f'Split: {num_train} train, {num_val} val, {num_test} test')
    
    test_files = annotation_files[:num_test]
    val_files = annotation_files[num_test:num_test + num_val]
    train_files = annotation_files[num_test + num_val:]
    
    def process_split(files, split_name):
        images_split_dir = yolo_dataset_path / 'images' / split_name
        labels_split_dir = yolo_dataset_path / 'labels' / split_name
        
        for ann_file in files:
            tree = ET.parse(ann_file)
            root = tree.getroot()
            
            filename = root.find('filename').text
            img_path = images_dir / filename
            
            size = root.find('size')
            img_width = int(size.find('width').text)
            img_height = int(size.find('height').text)
            
            dest_img_path = images_split_dir / filename
            if img_path.exists():
                shutil.copy2(img_path, dest_img_path)
            else:
                print(f'Warning: Image not found: {img_path}')
                continue
            
            label_path = labels_split_dir / (img_path.stem + '.txt')
            
            with open(label_path, 'w') as f:
                for obj in root.findall('object'):
                    if int(obj.find('difficult').text) == 1:
                        continue
                    
                    class_name = obj.find('name').text
                    bbox_elem = obj.find('bndbox')
                    xmin = int(bbox_elem.find('xmin').text)
                    ymin = int(bbox_elem.find('ymin').text)
                    xmax = int(bbox_elem.find('xmax').text)
                    ymax = int(bbox_elem.find('ymax').text)
                    
                    x_center, y_center, width, height = voc_to_yolo(
                        (xmin, ymin, xmax, ymax), img_width, img_height
                    )
                    
                    class_idx = class_to_idx[class_name]
                    f.write(f'{class_idx} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n')
    
    print('Processing train split...')
    process_split(train_files, 'train')
    print('Done with train split')
    
    print('Processing val split...')
    process_split(val_files, 'val')
    print('Done with val split')
    
    print('Processing test split...')
    process_split(test_files, 'test')
    print('Done with test split')
    
    dataset_config = {
        'path': str(yolo_dataset_path.absolute()),
        'train': 'images/train',
        'val': 'images/val',
        'test': 'images/test',
        'nc': len(sorted_classes),
        'names': idx_to_class
    }
    
    dataset_yaml_path = yolo_dataset_path / 'dataset.yaml'
    with open(dataset_yaml_path, 'w') as f:
        yaml.dump(dataset_config, f, default_flow_style=False)
    
    print(f'Dataset configuration saved to: {dataset_yaml_path}')
    print('Conversion complete!')
    
    return yolo_dataset_path, dataset_yaml_path, sorted_classes

In [ ]:
dataset_yaml = YOLO_DATASET_PATH / 'dataset.yaml'

if dataset_yaml.exists():
    print('YOLO dataset already exists. Skipping conversion.')
else:
    print('Converting Pascal VOC dataset to YOLO format...')
    yolo_path, yaml_path, classes = convert_voc_to_yolo(
        ORIGINAL_DATASET_PATH,
        YOLO_DATASET_PATH,
        test_split=0.2,
        val_split=0.1
    )
    print(f'YOLO dataset created at: {yolo_path}')
    print(f'Dataset YAML: {yaml_path}')
    print(f'Number of classes: {len(classes)}')

## Verify YOLO Dataset Structure

In [ ]:
yolo_images_dir = YOLO_DATASET_PATH / 'images'
yolo_labels_dir = YOLO_DATASET_PATH / 'labels'

print('YOLO Dataset Structure:')
for split in ['train', 'val', 'test']:
    split_images = list((yolo_images_dir / split).glob('*'))
    split_labels = list((yolo_labels_dir / split).glob('*'))
    print(f'  {split}: {len(split_images)} images, {len(split_labels)} label files')

with open(YOLO_DATASET_PATH / 'dataset.yaml', 'r') as f:
    config = yaml.safe_load(f)
    print('Dataset configuration:')
    print(yaml.dump(config, default_flow_style=False))

## Load the YOLOv8s model

In [ ]:
from ultralytics import YOLO

# Load model
model = YOLO(MODEL_NAME)
print(f'Model loaded: {MODEL_NAME}')

## Verify dataset before training

In [ ]:
yolo_yaml = YOLO_DATASET_PATH / 'dataset.yaml'
if not yolo_yaml.exists():
    raise FileNotFoundError(
        f'YOLO dataset.yaml not found at {yolo_yaml}.\n'
        f'Please run the conversion cells first.\n'
        f'Original dataset: {ORIGINAL_DATASET_PATH}\n'
        f'Converted dataset: {YOLO_DATASET_PATH}'
    )

print(f'Training with YOLO dataset: {yolo_yaml}')
print(f'Using device: {TRAINING_DEVICE}')

## Train the model

In [ ]:
results = model.train(
    data=str(YOLO_DATASET_PATH / 'dataset.yaml'),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    device=TRAINING_DEVICE,
    name='yolov8s_top10',
    workers=8,
    verbose=True,
    seed=SEED
)

## Evaluate the trained model

In [ ]:
best_model_path = OUTPUT_DIR / 'yolov8s_top10' / 'weights' / 'best.pt'
print(f'Best model path: {best_model_path}')

if best_model_path.exists():
    trained_model = YOLO(best_model_path)
    
    metrics = trained_model.val(
        data=str(YOLO_DATASET_PATH / 'dataset.yaml'),
        batch=BATCH_SIZE,
        imgsz=IMGSZ,
        device=TRAINING_DEVICE,
        split='val'
    )
    
    print('Evaluation metrics:')
    print(metrics)

## Make predictions on test images

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

test_images_dir = YOLO_DATASET_PATH / 'images' / 'test'
test_images = list(test_images_dir.glob('*.jpg'))[:5]

if len(test_images) == 0:
    test_images = list(test_images_dir.glob('*.png'))[:5]

print(f'Found {len(test_images)} test images')

if len(test_images) > 0 and best_model_path.exists():
    trained_model = YOLO(best_model_path)
    
    for img_path in test_images:
        print(f'Predicting on: {img_path.name}')
        
        results = trained_model(img_path, conf=0.5, device=TRAINING_DEVICE)
        
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        ax.imshow(img)
        
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cls_id = int(box.cls[0])
                cls_name = trained_model.names[cls_id]
                
                rect = plt.Rectangle(
                    (x1, y1), x2 - x1, y2 - y1,
                    fill=False,
                    color='red',
                    linewidth=2
                )
                ax.add_patch(rect)
                
                ax.text(
                    x1, y1 - 5,
                    f'{cls_name} {conf:.2f}',
                    color='red',
                    fontsize=12,
                    bbox=dict(facecolor='white', alpha=0.7)
                )
        
        ax.set_title(img_path.name)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        
        for result in results:
            print(f'  Found {len(result.boxes)} objects')
            for box in result.boxes:
                cls_id = int(box.cls[0])
                cls_name = trained_model.names[cls_id]
                conf = float(box.conf[0])
                print(f'    {cls_name}: {conf:.3f}')

## Save and export the trained model

In [ ]:
if best_model_path.exists():
    trained_model = YOLO(best_model_path)
    
    onnx_path = OUTPUT_DIR / 'yolov8s_top10' / 'weights' / 'model.onnx'
    trained_model.export(format='onnx', imgsz=IMGSZ)
    print(f'Model exported to ONNX format: {onnx_path}')
    
    run_dir = str(OUTPUT_DIR / 'yolov8s_top10')
    print(f'Training complete! Model saved to: {best_model_path}')
    print(f'Run directory: {run_dir}')